In [ ]:
from google.colab import files
uploaded = files.upload()


Saving MZVAV-1.csv to MZVAV-1 (1).csv
Saving MZVAV-2-1.csv to MZVAV-2-1.csv
Saving MZVAV-2-2.csv to MZVAV-2-2.csv


In [ ]:
!rm "MZVAV-1 (1).csv"



In [ ]:
!ls

MZVAV-1.csv  MZVAV-2-1.csv  MZVAV-2-2.csv  sample_data


## Installation

In [ ]:
!pip install numpy pandas scikit-learn matplotlib torch tqdm


In [ ]:
%%writefile hvac_fewshot_exact.py
#!/usr/bin/env python3

import os, argparse, json, random
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, confusion_matrix
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# reproducibility
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

SEQ_LEN = 10

# ---------- helpers ----------
def parse_datetime_column(df):
    for c in df.columns:
        if any(k in c.lower() for k in ["time","date"]):
            try:
                return pd.to_datetime(df[c]), c
            except: pass
    return None,None

def make_day_ids_from_datetime(dt):
    # Works for Series or DatetimeIndex
    if isinstance(dt, pd.Series):
        return (dt - dt.min()).dt.days.astype(int)
    elif isinstance(dt, pd.DatetimeIndex):
        return ((dt - dt.min()).days).astype(int)
    else:
        raise ValueError("dt must be a pandas Series or DatetimeIndex")

def sliding_sequences(a,L=SEQ_LEN):
    if len(a)<L: return np.empty((0,L,a.shape[1]))
    return np.stack([a[i-L+1:i+1] for i in range(L-1,len(a))])

def build_sequences_from_df(df, feats, label, L=SEQ_LEN, day_ids=None):
    X = df[feats].values
    y = df[label].values
    seqs = sliding_sequences(X,L)
    labels = y[L-1:]
    days = np.zeros_like(labels) if day_ids is None else day_ids[L-1:]
    return seqs, labels.astype(int), days

class SeqDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)  # ensure tensor
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        if self.y is not None:
            return self.X[i], self.y[i]
        else:
            return self.X[i]


# ---------- model ----------
class LSTMAE(nn.Module):
    def __init__(self,input_dim,h1=8,h2=4,drop=0.2):
        super().__init__()
        self.e1 = nn.LSTM(input_dim,h1,batch_first=True)
        self.e2 = nn.LSTM(h1,h2,batch_first=True)
        self.d1 = nn.LSTM(h2,h1,batch_first=True)
        self.d2 = nn.LSTM(h1,input_dim,batch_first=True)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(drop)
    def forward(self,x):
        x,_ = self.e1(x); x=self.drop(x)
        x,_ = self.e2(x); x=self.drop(x)
        x,_ = self.d1(x); x=self.drop(x)
        x,_ = self.d2(x)
        return self.relu(x)

def train_one_epoch(m,loader,opt,crit,dev):
    m.train(); s=0;n=0
    for b in loader:
        if isinstance(b,tuple): b=b[0]
        b=b.to(dev); opt.zero_grad(); out=m(b); loss=crit(out,b); loss.backward(); opt.step()
        s+=loss.item()*b.size(0); n+=b.size(0)
    return s/max(1,n)

def recon_errors(m, loader, dev):
    m.eval()
    errs = []
    labs = []
    with torch.no_grad():
        for batch in loader:
            # handle tuple (X, y) or just X
            if isinstance(batch, (list, tuple)) and len(batch) == 2 and isinstance(batch[0], torch.Tensor):
                X, y = batch
                X = X.to(dev)
            else:
                X = batch
                if isinstance(X, list):
                    X = torch.stack(X).to(dev)  # convert list of tensors -> batch tensor
                else:
                    X = X.to(dev)
                y = None
            r = m(X)
            e = ((r - X)**2).mean((1, 2)).cpu().numpy()
            errs += e.tolist()
            if y is not None:
                labs += y.cpu().numpy().tolist()
    return np.array(errs), (np.array(labs) if labs else None)


# ---------- pipeline ----------
def run(pretrain_csv, sim_csvs, k_days=3, trials=5, pretrain_epochs=100, finetune_epochs=10,
        batch=128, lr=1e-4, device='cuda', outdir='results'):

    os.makedirs(outdir,exist_ok=True)
    dev = torch.device(device if torch.cuda.is_available() and device=='cuda' else 'cpu')
    print("Device:",dev)

    # --- load real data
    df_real = pd.read_csv(pretrain_csv)
    dt_real, dt_col = parse_datetime_column(df_real)
    day_real = make_day_ids_from_datetime(dt_real)
    label_col = [c for c in df_real.columns if 'fault' in c.lower()][0]

    # features in real data
    feats = [c for c in df_real.columns if c not in [dt_col,label_col]]

    # --- load sim data
    frames = []; off=0
    for f in sim_csvs:
        df = pd.read_csv(f)
        dt, dtc = parse_datetime_column(df)
        if dt is None:
            df['_day'] = np.arange(len(df))//1440 + off
            off = df['_day'].max()+1
        else:
            df['_day'] = make_day_ids_from_datetime(dt)+off
            off = df['_day'].max()+1
        frames.append(df)
    sim = pd.concat(frames, ignore_index=True)

    # only use features that exist in both real & sim
    common_feats = [f for f in feats if f in sim.columns]

    real = df_real.copy()
    # scaling
    scaler = StandardScaler(); scaler.fit(df_real[common_feats])
    real[common_feats] = scaler.transform(real[common_feats])
    sim[common_feats] = scaler.transform(sim[common_feats])

    Xr, yr, dr = build_sequences_from_df(real, common_feats, label_col, SEQ_LEN, day_real)
    Xs, ys, ds = build_sequences_from_df(sim, common_feats, label_col, SEQ_LEN, sim['_day'].values)

    nom_idx = np.where(yr==0)[0]; Xr_nom = Xr[nom_idx]
    day_unique = np.unique(ds)
    day_anom = {d: ys[ds==d].sum()>0 for d in day_unique}
    normal_days = [d for d in day_unique if not day_anom[d]]
    anom_days = [d for d in day_unique if day_anom[d]]
    test_days = set(normal_days[:4]+anom_days)
    test_idx = np.where(np.isin(ds,list(test_days)))[0]
    pool_days = [d for d in day_unique if d not in test_days and not day_anom[d]]

    # --- pretrain
    model = LSTMAE(Xr_nom.shape[2]).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    loader = DataLoader(SeqDataset(Xr_nom), batch, True)
    for e in range(1, pretrain_epochs+1):
        l = train_one_epoch(model, loader, opt, crit, dev)
        if e%10==0 or e==1: print(f"pretrain {e}: {l:.6f}")
    torch.save(model.state_dict(), os.path.join(outdir,"pretrained.pt"))

    # --- trials
    res=[]
    for t in range(trials):
        print(f"\nTrial {t+1}/{trials}")
        chosen = np.random.choice(pool_days, size=min(k_days,len(pool_days)), replace=False)
        idx = np.where(np.isin(ds,chosen))[0]
        idx = idx[ys[idx]==0]; Xf = Xs[idx]
        m = LSTMAE(Xr_nom.shape[2]).to(dev)
        m.load_state_dict(torch.load(os.path.join(outdir,"pretrained.pt"), map_location=dev))
        opt = torch.optim.Adam(m.parameters(), lr=lr/5)
        if len(Xf)>0:
            for e in range(1, finetune_epochs+1):
                l = train_one_epoch(m, DataLoader(SeqDataset(Xf), batch, True), opt, crit, dev)
                if e%5==0 or e==finetune_epochs: print(f" fine {e}:{l:.6f}")

        tr_err,_ = recon_errors(m, DataLoader(SeqDataset(Xr_nom), batch), dev)
        ft_err,_ = recon_errors(m, DataLoader(SeqDataset(Xf), batch), dev) if len(Xf)>0 else (np.array([]),None)
        eta = max(tr_err.max(), ft_err.max() if len(ft_err)>0 else tr_err.max())
        Xt = Xs[test_idx]; yt = ys[test_idx]
        te_err, te_lab = recon_errors(m, DataLoader(SeqDataset(Xt, yt), batch), dev)
        fpr, tpr, _ = roc_curve(te_lab, te_err); AUC=auc(fpr,tpr)
        pred = (te_err>eta).astype(int); tn,fp,fn,tp=confusion_matrix(te_lab,pred).ravel()
        TPR=tp/(tp+fn); FPR=fp/(fp+tn)
        print(f" AUC={AUC:.4f}  TPR={TPR:.4f}  FPR={FPR:.4f}")
        res.append({'trial':t+1,'AUC':AUC,'TPR':TPR,'FPR':FPR})

    summary = {k:float(np.mean([r[k] for r in res])) for k in ['AUC','TPR','FPR']}
    print("\nMean results:", summary)
    with open(os.path.join(outdir,f"summary_k{k_days}.json"),'w') as f:
        json.dump({'summary':summary,'details':res}, f, indent=2)
    plt.plot(fpr, tpr, label=f"AUC={AUC:.3f}")
    plt.plot([0,1],[0,1],'--'); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.grid(); plt.show()

if __name__=="__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--pretrain", default="MZVAV-1.csv")
    parser.add_argument("--sim", nargs='+', default=["MZVAV-2-1.csv","MZVAV-2-2.csv"])
    parser.add_argument("--k", type=int, default=3)
    parser.add_argument("--trials", type=int, default=5)
    parser.add_argument("--device", default="cuda")
    args = parser.parse_args()
    run(args.pretrain, args.sim, k_days=args.k, trials=args.trials, device=args.device)


Overwriting hvac_fewshot_exact.py


In [ ]:
!python hvac_fewshot_exact.py --pretrain MZVAV-1.csv --sim MZVAV-2-1.csv MZVAV-2-2.csv --k 3 --trials 5 --device cuda


Device: cuda
pretrain 1: 0.811956
pretrain 10: 0.691853
pretrain 20: 0.671622
pretrain 30: 0.663537
pretrain 40: 0.613183
pretrain 50: 0.599270
pretrain 60: 0.595547
pretrain 70: 0.593479
pretrain 80: 0.592268
pretrain 90: 0.591499
pretrain 100: 0.590865

Trial 1/5
 fine 5:1302.127650
 fine 10:1301.446468
 AUC=0.7983  TPR=0.1670  FPR=0.0000

Trial 2/5
 fine 5:1608.576812
 fine 10:1608.549283
 AUC=0.7983  TPR=0.1682  FPR=0.0000

Trial 3/5
 fine 5:2033.657397
 fine 10:2033.624313
 AUC=0.7983  TPR=0.1898  FPR=0.0000

Trial 4/5
 fine 5:951.336073
 fine 10:950.153093
 AUC=0.7983  TPR=0.1844  FPR=0.0000

Trial 5/5
 fine 5:1116.328047
 fine 10:1115.699452
 AUC=0.7983  TPR=0.4467  FPR=0.1234

Mean results: {'AUC': 0.7983033632840844, 'TPR': 0.23121671066648145, 'FPR': 0.0246875}
Figure(640x480)


In [ ]:
"results/pretrained.pt"
"results/summary_k3.json"


'results/summary_k3.json'

In [ ]:
from google.colab import files
files.download('results/summary_k3.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>